<a href="https://colab.research.google.com/github/SABARISH-2008M/DAA-Lab/blob/main/DAA_experimentr_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import copy
import heapq
from itertools import permutations

INF = float("inf")


def reduce_matrix(mat):
    """Reduce matrix and return reduced matrix and reduction cost."""
    m = [row[:] for row in mat]
    n = len(m)
    cost = 0

    # Row reduction
    for i in range(n):
        row_min = min(m[i])
        if row_min and row_min != INF:
            cost += row_min
            m[i] = [x - row_min if x != INF else INF for x in m[i]]

    # Column reduction
    for j in range(n):
        col_min = min(m[i][j] for i in range(n))
        if col_min and col_min != INF:
            cost += col_min
            for i in range(n):
                if m[i][j] != INF:
                    m[i][j] -= col_min

    return m, cost


def tsp_branch_and_bound(cost_matrix, n):
    """TSP Solver using LC Branch and Bound with Matrix Reduction."""

    class Node:

        def __init__(self, matrix, path, cost, level, vertex):
            self.matrix = matrix
            self.path = path
            self.cost = cost
            self.level = level
            self.vertex = vertex

        def __lt__(self, other):
            return self.cost < other.cost

    # Prepare initial root matrix
    initial_matrix = [row[:] for row in cost_matrix]
    reduced_matrix, root_cost = reduce_matrix(initial_matrix)

    pq = []
    root = Node(reduced_matrix, [0], root_cost, 0, 0)
    heapq.heappush(pq, root)

    best_cost = INF
    best_path = []

    while pq:
        curr = heapq.heappop(pq)

        if curr.cost >= best_cost:
            continue

        if curr.level == n - 1:
            # Complete the tour back to source (vertex 0)
            final_cost = curr.cost + curr.matrix[curr.vertex][0]
            if final_cost < best_cost:
                best_cost = final_cost
                best_path = curr.path + [0]
            continue

        for v in range(n):
            if v not in curr.path and curr.matrix[curr.vertex][v] != INF:
                # Copy parent matrix
                child_mat = [row[:] for row in curr.matrix]

                # Set row of curr.vertex and col of v to INF
                for k in range(n):
                    child_mat[curr.vertex][k] = INF
                    child_mat[k][v] = INF

                # Set back-edge to INF to prevent returning early
                child_mat[v][0] = INF

                # Reduce child matrix
                reduced_child, red_cost = reduce_matrix(child_mat)
                edge_weight = curr.matrix[curr.vertex][v]
                child_cost = curr.cost + edge_weight + red_cost

                if child_cost < best_cost:
                    child_node = Node(
                        reduced_child,
                        curr.path + [v],
                        child_cost,
                        curr.level + 1,
                        v,
                    )
                    heapq.heappush(pq, child_node)

    return best_path, best_cost


def tsp_brute_force(cost, n):
    """Brute force for verification"""
    cities = list(range(1, n))
    best_cost = INF
    best_path = None

    for perm in permutations(cities):
        path = [0] + list(perm) + [0]
        c = sum(cost[path[i]][path[i + 1]] for i in range(n))
        if c < best_cost:
            best_cost = c
            best_path = path

    return best_path, best_cost


# --- Main Execution ---
if __name__ == "__main__":
    # 5-city cost matrix
    cost = [
        [INF, 10, 8, 9, 7],
        [10, INF, 10, 5, 6],
        [8, 10, INF, 8, 9],
        [9, 5, 8, INF, 6],
        [7, 6, 9, 6, INF],
    ]
    n = 5
    cities = ["A", "B", "C", "D", "E"]

    # Solve using Branch and Bound & Brute Force Verification
    best_path, best_cost = tsp_branch_and_bound(cost, n)

    print("5-City TSP - Cost Matrix:")
    print(f"{'':>4}", " ".join(f"{c:>5}" for c in cities))
    for i, row in enumerate(cost):
        r = ["INF" if x == INF else str(x) for x in row]
        print(f"{cities[i]:>4}", " ".join(f"{v:>5}" for v in r))

    print(f"\nOptimal Tour: {' -> '.join(cities[i] for i in best_path)}")
    print(f"Minimum Cost: {best_cost}")

    print(f"\nPath verification:")
    for i in range(n):
        u, v = best_path[i], best_path[i + 1]
        print(f"  {cities[u]} -> {cities[v]}: cost = {cost[u][v]}")